# Chapter 3 – Mooring Design

**Project:** Floating Submerged Tunnel – Helsinki to Tallinn  
**Objective:** Design the tether/anchor system for the selected SFT concept.

---

## 3.1 Design Basis

| Parameter | Value | Source |
|-----------|-------|---------|
| Water depth (midpoint) | ~70 m | Baltic Sea charts |
| Design significant wave height H₁₀₀ | TBD m | Chapter 1 EVA |
| Design current speed V₁₀₀ | TBD m/s | Chapter 1 EVA |
| Tunnel diameter (outer) | TBD m | Chapter 2 |
| Submergence depth (top of tunnel) | TBD m | Chapter 2 |
| Buoyancy–weight ratio (BWR) | TBD | Design target |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# -----------------------------------------------------------------------
# Physical constants
# -----------------------------------------------------------------------
RHO_W = 1025.0   # seawater density [kg/m³]
RHO_AIR = 1.225  # air density [kg/m³]
G = 9.81         # gravitational acceleration [m/s²]

## 3.2 Tether Configuration

For the SFT tether-moored concept, vertical (or near-vertical) steel tethers connect the tunnel tube to gravity anchors on the seabed.


In [ ]:
# -----------------------------------------------------------------------
# Tunnel cross-section properties
# TODO: Update with selected design values from Chapter 2
# -----------------------------------------------------------------------
D_outer  = 14.0     # outer diameter [m]
D_inner  = 12.0     # inner diameter (roadway bore) [m]
t_shell  = 0.50     # concrete shell thickness [m]
L_span   = 30.0     # tether spacing along tunnel axis [m]
z_tunnel = -20.0    # depth of tunnel axis below MWL [m] (negative = below surface)
h_water  = 70.0     # water depth at site [m]

# Tether geometry
z_anchor = -h_water         # anchor depth (seabed)
L_tether = abs(z_tunnel - z_anchor)  # tether length [m]

# Cross-sectional areas
A_outer  = np.pi * D_outer**2 / 4
A_inner  = np.pi * D_inner**2 / 4
A_concrete = A_outer - A_inner

print(f"Tunnel outer area:     {A_outer:.2f} m²")
print(f"Tunnel concrete area:  {A_concrete:.2f} m²")
print(f"Tether length:         {L_tether:.1f} m")

## 3.3 Buoyancy & Weight Calculation


In [ ]:
# -----------------------------------------------------------------------
# Weights and buoyancy per unit length [kN/m]
# -----------------------------------------------------------------------
RHO_CONCRETE = 2400.0  # [kg/m³]
RHO_ASPHALT  = 2300.0  # road surface [kg/m³]
t_road = 0.10           # road surface thickness [m]

# Buoyancy force per unit length
F_buoy = RHO_W * G * A_outer / 1e3   # [kN/m]

# Self-weight of concrete shell per unit length
W_concrete = RHO_CONCRETE * G * A_concrete / 1e3  # [kN/m]

# Road surface weight (approximate – two bores)
W_road = RHO_ASPHALT * G * (2 * D_inner * t_road) / 1e3  # [kN/m]

# Net upward force per unit length (positive = upward)
F_net_per_m = F_buoy - W_concrete - W_road

# Buoyancy-Weight Ratio
BWR = F_buoy / (W_concrete + W_road)

print(f"Buoyancy force:        {F_buoy:.1f} kN/m")
print(f"Self-weight (concrete):{W_concrete:.1f} kN/m")
print(f"Road surface weight:   {W_road:.1f} kN/m")
print(f"Net upward force:      {F_net_per_m:.1f} kN/m")
print(f"BWR:                   {BWR:.3f}")

## 3.4 Tether Design Loads


In [ ]:
# -----------------------------------------------------------------------
# Tether tension under quasi-static loading
# -----------------------------------------------------------------------
# Net buoyancy per span → tether pre-tension
T_pretension = F_net_per_m * L_span   # [kN] per tether

# Wave-induced horizontal force (simplified Morison, regular wave)
# H_design [m], T_wave [s] – from Chapter 1 EVA (100-yr return period)
H_design = 5.0    # TODO: replace with EVA result
T_wave   = 9.0    # TODO: replace with EVA result
U_current = 0.5   # 100-yr current [m/s] TODO: from EVA

omega = 2 * np.pi / T_wave
k = omega**2 / G   # deep-water approximation
C_D = 1.0          # drag coefficient (circular cross-section)
C_M = 2.0          # inertia coefficient

# Particle velocity amplitude at tunnel axis depth
U_wave = (H_design / 2) * omega * np.exp(k * z_tunnel)  # [m/s]
A_wave = (H_design / 2) * omega**2 * np.exp(k * z_tunnel)  # [m/s²] acceleration

# Morison force per unit length at max drag + max inertia (envelope)
f_drag    = 0.5 * RHO_W * C_D * D_outer * (U_wave + U_current)**2 / 1e3  # [kN/m]
f_inertia = RHO_W * C_M * A_outer * A_wave / 1e3                          # [kN/m]
f_total   = f_drag + f_inertia  # [kN/m]

# Horizontal tether force component per span
T_horizontal = f_total * L_span  # [kN]

# Design tether tension (combined vertical pre-tension + horizontal load)
T_design = np.sqrt(T_pretension**2 + T_horizontal**2)

print(f"Tether pre-tension:     {T_pretension:.0f} kN")
print(f"Wave particle velocity: {U_wave:.3f} m/s  (at z={z_tunnel} m)")
print(f"Drag force:             {f_drag:.2f} kN/m")
print(f"Inertia force:          {f_inertia:.2f} kN/m")
print(f"Total horizontal load:  {T_horizontal:.0f} kN per tether")
print(f"Design tether tension:  {T_design:.0f} kN")

## 3.5 Tether Sizing


In [ ]:
# -----------------------------------------------------------------------
# Steel tether cross-section (solid bar or hollow tube)
# -----------------------------------------------------------------------
SIGMA_Y = 355e3    # yield strength [kN/m²] – S355 structural steel
GAMMA_M = 1.1      # material safety factor (DNVGL-OS-E301)
GAMMA_F = 1.3      # load safety factor

T_ult_required = T_design * GAMMA_F  # [kN]
A_tether_min   = T_ult_required / (SIGMA_Y / GAMMA_M)  # [m²]
d_tether_min   = 2 * np.sqrt(A_tether_min / np.pi) * 1e3  # [mm]

print(f"Factored design tension:  {T_ult_required:.0f} kN")
print(f"Min tether cross-section: {A_tether_min*1e4:.1f} cm²")
print(f"Min tether diameter:      {d_tether_min:.0f} mm  (solid bar)")

# Actual chosen tether (round up to standard size)
d_chosen_mm = np.ceil(d_tether_min / 10) * 10  # round up to nearest 10 mm
A_chosen    = np.pi * (d_chosen_mm / 1e3)**2 / 4
sigma_actual = T_ult_required / A_chosen  # [kN/m²]
UC = sigma_actual / (SIGMA_Y / GAMMA_M)   # utilisation factor

print(f"\nChosen tether diameter: {d_chosen_mm:.0f} mm")
print(f"Cross-sectional area:   {A_chosen*1e4:.1f} cm²")
print(f"Utilisation factor:     {UC:.3f}  (≤ 1.0 required)")

## 3.6 Catenary / Taut-Leg Configuration Comparison


In [ ]:
# -----------------------------------------------------------------------
# Catenary profile (illustrative, horizontal load at fairlead)
# -----------------------------------------------------------------------
def catenary_profile(H, w, s_max=None, n=200):
    """Catenary shape for a mooring line.
    H   – horizontal tension at fairlead [kN]
    w   – submerged weight per unit length [kN/m]
    s_max – arc length of line [m] (defaults to tether length)
    Returns x, z arrays.
    """
    if s_max is None:
        s_max = L_tether
    s = np.linspace(0, s_max, n)
    x = (H / w) * np.sinh(w * s / H)
    z = (H / w) * (np.cosh(w * s / H) - 1)
    return x, z

w_steel = 7850 * G * A_chosen / 1e3  # submerged weight/m [kN/m] (approx)

fig, ax = plt.subplots(figsize=(8, 5))
for H_horiz in [50, 100, 200, 500]:
    x, z = catenary_profile(H_horiz, w_steel)
    ax.plot(x, -z, label=f'H = {H_horiz} kN')

ax.set_xlabel('Horizontal offset [m]')
ax.set_ylabel('Depth [m]')
ax.set_title('Catenary Mooring Profile (illustrative)')
ax.invert_yaxis()
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 3.7 Summary

| Parameter | Value |
|-----------|-------|
| BWR | *(fill in)* |
| Tether diameter | *(fill in)* mm |
| Tether spacing | *(fill in)* m |
| Design tension | *(fill in)* kN |
| Utilisation factor | *(fill in)* |

---
*Proceed to Chapter 4 – Numerical Modelling & Dynamical Assessment*